# Structure des 20 arbres de la taxonomie (v3)

**Objectif** — comprendre la *forme* réelle des données de JB (livraison v3, `structure/taxonomy_32.json`)
pour en déduire une **logique de navigation**. On répond à trois questions simples :

1. Combien d'arbres, et de quelle **taille** ?
2. Jusqu'où **descendent**-ils (profondeur, « aller jusqu'au bout ») ?
3. La « généalogie » est-elle **riche** (beaucoup d'enfants) ou **pauvre** (chaînes) ?

Et on conclut sur **comment naviguer** dans ces 20 arbres.

> Tout est recalculé ici depuis les fichiers bruts — le notebook est **autonome** (indépendant du POC Gradio).

## 0 · Nettoyage : qu'est-ce qu'un topic « propre » ?

Avant de compter les arbres, on écarte le bruit. Un topic est **propre** s'il est :

- **relié** à un arbre — pas *isolé* (ni parent, ni enfant) ;
- **sans cycle** — aucun ancêtre n'est aussi son propre descendant.

En v3 c'est très propre (à comparer aux **78 cycles / 1029 isolés** de la v1).

In [1]:
import json
from collections import Counter, defaultdict, deque
from pathlib import Path

import pandas as pd

DATA = Path("analysis_v3")
topics = json.loads((DATA / "structure/taxonomy_32.json").read_text())["topics"]
docs = json.loads((DATA / "label/instances.json").read_text())["documents"]

by_id = {t["id"]: t for t in topics}
by_name = {t["name"]: t for t in topics}        # les noms sont uniques en v3

# détections « propres » à chaque topic = occurrences directes repérées dans les textes
own = defaultdict(int)
for d in docs:
    for lab in d["labels"]:
        own[lab["name"]] += 1
own = dict(own)

# en v3 le parent est un UUID → on le résout vers le NOM du parent
parent_nom = {t["name"]: (by_id[t["parent"]]["name"] if t["parent"] in by_id else None)
              for t in topics}

print(f"{len(topics)} topics · {len(docs)} documents analysés")

4846 topics · 1524 documents analysés


In [2]:
# --- propre = relié ET sans cycle ---
est_parent = {p for p in parent_nom.values() if p}
isoles = {n for n in by_name if not parent_nom[n] and n not in est_parent}


def _cyclique(nom, vus=None):
    vus = vus or set()
    if nom in vus:
        return True
    vus.add(nom)
    p = parent_nom.get(nom)
    return _cyclique(p, vus) if p and p in by_name else False


cycliques = {n for n in by_name if _cyclique(n)}
propre = set(by_name) - isoles - cycliques

print(f"topics propres : {len(propre)} / {len(by_name)}")
print(f"  écartés → isolés : {len(isoles)} · cycliques : {len(cycliques)}")

topics propres : 4835 / 4846
  écartés → isolés : 11 · cycliques : 0


## 1 · 20 arbres, mais de quelle taille ?

Un **arbre** = un ensemble de topics reliés par parent→enfant, avec une **racine**
(topic propre sans parent propre) qui porte au moins une détection.

On construit la structure, puis on mesure chaque arbre.

In [3]:
parent_de = {n: parent_nom[n] for n in propre if parent_nom[n] in propre}
enfants = defaultdict(list)
for n, p in parent_de.items():
    enfants[p].append(n)

_memo = {}
def rec(n):                     # détections agrégées sur tout le sous-arbre
    if n not in _memo:
        _memo[n] = own.get(n, 0) + sum(rec(c) for c in enfants[n])
    return _memo[n]

def niv(n):                     # le champ `level` de JB (0 feuille … 7 racine)
    return by_name[n]["level"]

def sous_arbre(r):
    vus, f = set(), deque([r])
    while f:
        x = f.popleft()
        if x in vus:
            continue
        vus.add(x)
        f.extend(enfants[x])
    return vus

def profondeur(r):              # nb d'arêtes racine → feuille la plus lointaine
    d, f, mx = {r: 0}, deque([r]), 0
    while f:
        x = f.popleft()
        for c in enfants[x]:
            d[c] = d[x] + 1
            mx = max(mx, d[c])
            f.append(c)
    return mx

ROOTS = sorted((n for n in propre if n not in parent_de and rec(n) > 0),
               key=rec, reverse=True)
print(f"{len(ROOTS)} arbres (racines propres portant au moins une détection)")

20 arbres (racines propres portant au moins une détection)


In [4]:
tab = pd.DataFrame([{
    "racine": r,
    "niv. racine": niv(r),
    "nœuds": len(sous_arbre(r)),
    "profondeur": profondeur(r),
    "feuilles": sum(1 for n in sous_arbre(r) if not enfants[n]),
    "détections": rec(r),
} for r in ROOTS])
tab

,racine,niv. racine,nœuds,profondeur,feuilles,détections
0,sciences humaines et sociales,7,3667,7,3224,138713
1,sciences sociales,6,937,6,742,39315
2,"société, politique et organisation collective",4,115,4,86,5799
3,justice sociale et économique,2,7,2,5,586
4,méthodes d'analyse et d'évaluation,4,23,4,11,572
5,développement et transformation des sociétés,4,8,4,3,337
6,problèmes sociaux et économiques,1,5,1,4,313
7,réformes et politiques publiques,2,9,2,7,278
8,données et information,2,10,2,7,270
9,philosophie du travail,3,6,3,3,240


In [5]:
TOT = tab["détections"].sum()
gros = tab[tab["nœuds"] >= 100]
petits = tab[tab["nœuds"] < 100]
print(f"{len(gros):2d} arbres « substantiels » (≥100 nœuds) → {gros['détections'].sum():>6d} détections "
      f"= {round(100 * gros['détections'].sum() / TOT)} %")
print(f"{len(petits):2d} fragments            (<100 nœuds) → {petits['détections'].sum():>6d} détections "
      f"= {round(100 * petits['détections'].sum() / TOT)} %")
g = tab.iloc[0]
print(f"\nÀ lui seul, « {g['racine']} » = {g['nœuds']} nœuds "
      f"= {round(100 * g['nœuds'] / len(propre))} % de toute la taxonomie propre.")

 3 arbres « substantiels » (≥100 nœuds) → 183827 détections = 98 %
17 fragments            (<100 nœuds) →   3480 détections = 2 %

À lui seul, « sciences humaines et sociales » = 3667 nœuds = 76 % de toute la taxonomie propre.


### Lecture du tableau

Deux mondes cohabitent :

- **2 à 3 géants** — `sciences humaines et sociales` (3667 nœuds), `sciences sociales` (937),
  `société, politique et organisation collective` (115) — concentrent **98 % des détections** ;
  le premier pèse à lui seul **76 %** de la taxonomie propre.
- **17 fragments** (2 à 23 nœuds) qui, ensemble, ne portent que **2 %** des détections.

« 20 arbres » est donc trompeur : c'est **1 forêt dense + une poussière de brindilles**.

## 2 · Jusqu'où descendent-ils ? — trois lectures de « jusqu'au bout »

Ta question « combien vont jusqu'au bout ? » n'a pas **une** réponse : « le bout » peut vouloir
dire trois choses. On mesure les trois.

In [6]:
atteint_0 = sum(min(niv(n) for n in sous_arbre(r)) == 0 for r in ROOTS)
traverse_tout = sum(profondeur(r) >= 7 for r in ROOTS)
vraie_genea = sum(profondeur(r) >= 4 for r in ROOTS)

print("« Aller jusqu'au bout » a trois lectures :\n")
print(f"  (a) atteindre une feuille de niveau 0 ......... {atteint_0}/20 arbres")
print(f"  (b) traverser les 8 niveaux (profondeur ≥ 7) .. {traverse_tout}/20 arbres")
print(f"  (c) avoir ≥ 4 générations (profondeur ≥ 4) .... {vraie_genea}/20 arbres")

print("\nRépartition des profondeurs (en nombre d'arêtes) :")
for p, c in sorted(Counter(profondeur(r) for r in ROOTS).items()):
    print(f"  profondeur {p} : {c} arbre(s)")

« Aller jusqu'au bout » a trois lectures :

  (a) atteindre une feuille de niveau 0 ......... 20/20 arbres
  (b) traverser les 8 niveaux (profondeur ≥ 7) .. 1/20 arbres
  (c) avoir ≥ 4 générations (profondeur ≥ 4) .... 6/20 arbres

Répartition des profondeurs (en nombre d'arêtes) :
  profondeur 1 : 6 arbre(s)
  profondeur 2 : 7 arbre(s)
  profondeur 3 : 1 arbre(s)
  profondeur 4 : 4 arbre(s)
  profondeur 6 : 1 arbre(s)
  profondeur 7 : 1 arbre(s)


### Ce qu'il faut retenir

- **20/20** atteignent une feuille de niveau 0 → cette lecture ne discrimine rien (inutile).
- **1 seul** arbre traverse réellement les 8 niveaux.
- **6** arbres ont une vraie généalogie (≥ 4 générations) ; **6** sont **plats** (profondeur 1 :
  la racine et ses feuilles, rien entre les deux).

« Aller jusqu'au bout » de façon *intéressante* (≥ 4 générations) ne concerne qu'**un tiers** des arbres.

## 3 · Piège : le champ `level` ne dit **pas** la profondeur

On pourrait croire que `level` = position dans l'arbre (7 racine … 0 feuille). **C'est faux.**
`level` est un **rang d'abstraction absolu**, attribué topic par topic, **indépendant de la structure**.
Deux symptômes :

In [7]:
print("Niveau (`level`) des 20 racines — une vraie racine devrait être au niveau max :")
for lvl, c in sorted(Counter(niv(r) for r in ROOTS).items(), reverse=True):
    print(f"  racines déclarées au niveau {lvl} : {c}")

sauts = Counter(niv(parent_de[n]) - niv(n) for n in parent_de)
tot_e = sum(sauts.values())
print(f"\nSauts de niveau parent→enfant (idéal = +1) sur {tot_e} arêtes :")
for s in sorted(sauts):
    print(f"  écart {s:+d} niveau(x) : {sauts[s]:5d} arêtes ({round(100 * sauts[s] / tot_e)} %)")

Niveau (`level`) des 20 racines — une vraie racine devrait être au niveau max :
  racines déclarées au niveau 7 : 1
  racines déclarées au niveau 6 : 1
  racines déclarées au niveau 4 : 4
  racines déclarées au niveau 3 : 1
  racines déclarées au niveau 2 : 7
  racines déclarées au niveau 1 : 6

Sauts de niveau parent→enfant (idéal = +1) sur 4815 arêtes :
  écart +1 niveau(x) :  4078 arêtes (85 %)
  écart +2 niveau(x) :   594 arêtes (12 %)
  écart +3 niveau(x) :   137 arêtes (3 %)
  écart +4 niveau(x) :     6 arêtes (0 %)


### Conséquence directe pour la navigation

- Des racines sont déclarées à **tous les niveaux de 1 à 7** : un « niveau 1 » sans parent est
  soit un fragment orphelin, soit une racine mal cotée. `level` et place dans l'arbre **se contredisent**.
- **~15 % des arêtes sautent des générations** (un enfant n'est pas toujours `parent − 1`).

👉 **On ne peut pas naviguer avec `level`.** Il faut suivre les **arêtes réelles** (parent→enfant),
c'est-à-dire la **profondeur structurelle** — exactement ce que fait la cascade du POC. `level` ne sert
plus qu'à la **couleur** (repère visuel), pas au déplacement.

## 4 · Généalogie riche ou pauvre ?

« Une généalogie à peu d'enfants, c'est nul » — mesurons la **ramification** (enfants par nœud interne)
et la part de **chaînes** (nœuds à un seul enfant, qui n'offrent aucun choix quand on déroule).

In [8]:
import statistics

internes = [n for n in propre if enfants[n]]
br = [len(enfants[n]) for n in internes]
print(f"nœuds internes : {len(internes)} · feuilles : {len(propre) - len(internes)}")
print(f"enfants par nœud interne : médiane {int(statistics.median(br))}, "
      f"moyenne {statistics.mean(br):.1f}, max {max(br)}")
chaines = sum(1 for b in br if b == 1)
print(f"nœuds « en chaîne » (1 seul enfant) : {chaines} = {round(100 * chaines / len(internes))} % des internes")

nœuds internes : 714 · feuilles : 4121
enfants par nœud interne : médiane 3, moyenne 6.7, max 160
nœuds « en chaîne » (1 seul enfant) : 163 = 23 % des internes


### Nuance

- En **moyenne** ça ramifie bien (médiane 3 enfants, jusqu'à 160), surtout dans les 2 géants.
- Mais **~1 nœud interne sur 4 est une chaîne** : dérouler n'y offre aucun choix, c'est de la
  profondeur « vide ». Chaînes + fragments plats = l'impression de généalogie pauvre que tu ressens.

## 5 · Synthèse : classer les 20 arbres par navigabilité

On range chaque arbre selon **taille × profondeur** :

- **A** — hiérarchie profonde (≥ 100 nœuds *et* ≥ 4 générations) : vrai *drill-down*.
- **B** — arbuste (≥ 3 générations) : navigable mais court.
- **C** — buisson (2 niveaux).
- **D** — plat (racine + feuilles) : aucune généalogie à dérouler.

In [9]:
def classe(r):
    n, p = len(sous_arbre(r)), profondeur(r)
    if n >= 100 and p >= 4:
        return "A · hiérarchie profonde"
    if p >= 3:
        return "B · arbuste"
    if p == 2:
        return "C · buisson (2 niveaux)"
    return "D · plat (racine + feuilles)"

tab["classe"] = tab["racine"].map(classe)
resume = (tab.groupby("classe")
             .agg(arbres=("racine", "size"), détections=("détections", "sum"))
             .reset_index())
resume["% détections"] = (100 * resume["détections"] / TOT).round().astype(int)
resume

,classe,arbres,détections,% détections
0,A · hiérarchie profonde,3,183827,98
1,B · arbuste,4,1381,1
2,C · buisson (2 niveaux),7,1355,1
3,D · plat (racine + feuilles),6,744,0


In [10]:
# le détail : quels arbres dans chaque classe
tab[["racine", "classe", "nœuds", "profondeur", "détections"]].sort_values(
    ["classe", "détections"], ascending=[True, False])

,racine,classe,nœuds,profondeur,détections
0,sciences humaines et sociales,A · hiérarchie profonde,3667,7,138713
1,sciences sociales,A · hiérarchie profonde,937,6,39315
2,"société, politique et organisation collective",A · hiérarchie profonde,115,4,5799
4,méthodes d'analyse et d'évaluation,B · arbuste,23,4,572
5,développement et transformation des sociétés,B · arbuste,8,4,337
9,philosophie du travail,B · arbuste,6,3,240
11,technologies et transformations numériques,B · arbuste,18,4,232
3,justice sociale et économique,C · buisson (2 niveaux),7,2,586
7,réformes et politiques publiques,C · buisson (2 niveaux),9,2,278
8,données et information,C · buisson (2 niveaux),10,2,270


## Conclusion — quelle logique de navigation ?

**Le constat**

- La forêt est **ultra-déséquilibrée** : 2–3 arbres portent 98 % du signal, 17 sont des fragments.
- Le champ `level` est **inexploitable pour se déplacer** (rangs absolus + 15 % d'arêtes qui sautent).
- La vraie profondeur navigable vit surtout dans **`sciences humaines et sociales`** (et un peu `sciences sociales`).

**La logique de navigation qui en découle**

1. **Naviguer par la structure, pas par `level`.** La cascade suit les arêtes parent→enfant
   (profondeur réelle) : c'est le bon choix, on le garde. `level` → couleur uniquement.
2. **Deux vitesses.** Mettre en avant les **2–3 racines substantielles** (triées par détections) ;
   présenter les **17 fragments à plat** (liste), sans drill-down inutile.
3. **Sauter les chaînes.** Un nœud à 1 seul enfant peut être franchi automatiquement à la descente
   (on s'arrête au prochain vrai embranchement) → navigation moins frustrante.

**À arbitrer avec JB (data quality)**

- Faut-il **rattacher les 18 racines mineures** sous les 2 géants ? (`sciences sociales` niv 6 sous
  `sciences humaines et sociales` niv 7 ?) — sinon la forêt reste éclatée artificiellement.
- **Normaliser** les sauts de niveau (15 % des arêtes) et les chaînes (23 % des internes) ?
- Confirmer que ces **20 racines** sont un choix volontaire, pas un artefact de découpage.